# LSTM Training for Price Prediction

This notebook demonstrates:
- Loading data and generating features with `FeaturePipeline`
- Creating sequence windows for LSTM input
- Training an LSTM model from `medusa.models.lstm`
- Evaluating model performance
- Generating predictions and comparing to actual values

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from medusa.features.pipeline import FeaturePipeline
from medusa.models.lstm import LSTMModel

try:
    import lightning as L
    HAS_LIGHTNING = True
except ImportError:
    HAS_LIGHTNING = False
    print("PyTorch Lightning not installed. Install with: pip install lightning")

## 1. Generate Synthetic Data with Trend

Create a price series with cyclical pattern for LSTM to learn.

In [ ]:
# Generate synthetic data with trend + cycle
np.random.seed(42)
n = 1000
dates = pd.date_range("2023-01-01", periods=n, freq="1h")

# Trend + seasonality + noise
t = np.arange(n)
trend = 0.05 * t
seasonal = 10 * np.sin(2 * np.pi * t / 50)  # 50-hour cycle
noise = np.random.normal(0, 2, n)
price = 100 + trend + seasonal + noise

# Generate OHLC
high = price * (1 + np.abs(np.random.normal(0, 0.005, n)))
low = price * (1 - np.abs(np.random.normal(0, 0.005, n)))
open_price = price + np.random.normal(0, 0.2, n)
volume = np.random.randint(1000, 10000, n)

df = pd.DataFrame(
    {
        "open": open_price,
        "high": high,
        "low": low,
        "close": price,
        "volume": volume,
    },
    index=dates,
)

print(f"Generated {len(df)} bars")
df.head()

In [ ]:
# Visualize
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["close"])
plt.title("Synthetic Price with Trend and Seasonality")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True)
plt.show()

## 2. Feature Engineering with Pipeline

Use `FeaturePipeline` to add indicators and prepare data.

In [ ]:
# Initialize feature pipeline
pipeline = FeaturePipeline(
    target_col="return_1",
    train_ratio=0.8,
)

# Run pipeline
data = pipeline.run(df)

print(f"\nFeatures: {len(data['feature_columns'])}")
print(f"Train samples: {len(data['x_train'])}")
print(f"Test samples: {len(data['x_test'])}")

## 3. Create Sequence Windows for LSTM

LSTM expects 3D input: (batch, sequence_length, features)

In [ ]:
def create_sequences(x, y, seq_length=20):
    """Create sliding windows for LSTM.
    
    Args:
        x: Feature array (samples, features)
        y: Target array (samples,)
        seq_length: Number of timesteps per sequence
        
    Returns:
        x_seq: (num_sequences, seq_length, features)
        y_seq: (num_sequences,)
    """
    x_seq, y_seq = [], []

    for i in range(len(x) - seq_length):
        x_seq.append(x[i:i+seq_length])
        y_seq.append(y[i+seq_length])  # Predict next value

    return np.array(x_seq), np.array(y_seq)

# Create sequences
SEQ_LENGTH = 20

x_train_seq, y_train_seq = create_sequences(
    data["x_train"].values,
    data["y_train"].values,
    SEQ_LENGTH
)
x_test_seq, y_test_seq = create_sequences(
    data["x_test"].values,
    data["y_test"].values,
    SEQ_LENGTH
)

print(f"Train sequences: {x_train_seq.shape}")
print(f"Test sequences: {x_test_seq.shape}")

In [ ]:
# Convert to PyTorch tensors
x_train_tensor = torch.FloatTensor(x_train_seq)
y_train_tensor = torch.FloatTensor(y_train_seq).unsqueeze(1)
x_test_tensor = torch.FloatTensor(x_test_seq)
y_test_tensor = torch.FloatTensor(y_test_seq).unsqueeze(1)

# Create DataLoaders
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## 4. Train LSTM Model

Initialize and train the model using PyTorch Lightning.

In [ ]:
if not HAS_LIGHTNING:
    print("Skipping training — PyTorch Lightning not installed")
else:
    # Initialize model
    input_size = x_train_seq.shape[2]  # Number of features

    model = LSTMModel(
        input_size=input_size,
        hidden_size=128,
        num_layers=2,
        dropout=0.2,
        output_size=1,
        learning_rate=1e-3,
    )

    # Initialize trainer
    trainer = L.Trainer(
        max_epochs=50,
        accelerator="auto",
        enable_progress_bar=True,
        log_every_n_steps=10,
    )

    # Train
    print("\nTraining LSTM...")
    trainer.fit(model, train_loader, test_loader)

## 5. Evaluate Model Performance

In [ ]:
if HAS_LIGHTNING:
    # Test performance
    trainer.test(model, test_loader)

    # Generate predictions
    model.eval()
    with torch.no_grad():
        y_pred_train = model(x_train_tensor).numpy().flatten()
        y_pred_test = model(x_test_tensor).numpy().flatten()

    y_train_actual = y_train_tensor.numpy().flatten()
    y_test_actual = y_test_tensor.numpy().flatten()

    # Compute metrics
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    train_mse = mean_squared_error(y_train_actual, y_pred_train)
    test_mse = mean_squared_error(y_test_actual, y_pred_test)
    test_mae = mean_absolute_error(y_test_actual, y_pred_test)
    test_r2 = r2_score(y_test_actual, y_pred_test)

    print("\n" + "="*60)
    print("MODEL PERFORMANCE")
    print("="*60)
    print(f"Train MSE: {train_mse:.6f}")
    print(f"Test MSE: {test_mse:.6f}")
    print(f"Test MAE: {test_mae:.6f}")
    print(f"Test R²: {test_r2:.4f}")
    print("="*60)

## 6. Visualize Predictions vs Actual

In [ ]:
if HAS_LIGHTNING:
    # Plot predictions vs actual
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Train set
    axes[0].plot(y_train_actual, label="Actual", alpha=0.7)
    axes[0].plot(y_pred_train, label="Predicted", alpha=0.7)
    axes[0].set_title("Train Set: Actual vs Predicted Returns")
    axes[0].set_xlabel("Sample")
    axes[0].set_ylabel("Return")
    axes[0].legend()
    axes[0].grid(True)

    # Test set
    axes[1].plot(y_test_actual, label="Actual", alpha=0.7)
    axes[1].plot(y_pred_test, label="Predicted", alpha=0.7)
    axes[1].set_title("Test Set: Actual vs Predicted Returns")
    axes[1].set_xlabel("Sample")
    axes[1].set_ylabel("Return")
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
if HAS_LIGHTNING:
    # Scatter plot: predicted vs actual
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test_actual, y_pred_test, alpha=0.5)
    plt.plot([-0.1, 0.1], [-0.1, 0.1], 'r--', label="Perfect prediction")
    plt.xlabel("Actual Return")
    plt.ylabel("Predicted Return")
    plt.title("Predicted vs Actual (Test Set)")
    plt.legend()
    plt.grid(True)
    plt.axis('equal')
    plt.show()

## 7. Directional Accuracy

Check if model can predict direction correctly (up/down).

In [ ]:
if HAS_LIGHTNING:
    # Directional accuracy
    direction_actual = (y_test_actual > 0).astype(int)
    direction_pred = (y_pred_test > 0).astype(int)

    directional_accuracy = (direction_actual == direction_pred).mean()

    print(f"\nDirectional Accuracy: {directional_accuracy:.2%}")
    print("(Above 50% means model is better than random for direction prediction)")

## 8. Hyperparameter Discussion

### Key hyperparameters for LSTM:

1. **`hidden_size`**: Size of LSTM hidden state (128-512 typical)
   - Larger = more capacity, but risk of overfitting

2. **`num_layers`**: Number of stacked LSTM layers (1-3 typical)
   - More layers = more complex patterns, but slower training

3. **`dropout`**: Dropout rate for regularization (0.1-0.5)
   - Higher = less overfitting, but may underfit

4. **`learning_rate`**: Adam optimizer step size (1e-4 to 1e-2)
   - Too high = unstable, too low = slow convergence

5. **`seq_length`**: Lookback window (10-50 typical)
   - Longer = more context, but harder to learn

### Next steps:
- Use Optuna to optimize these parameters (see `medusa.backtest.optimizer`)
- Try different architectures (GRU, Transformer, Temporal Fusion)
- Add more features (order flow, cross-asset correlations)